# Ethiopia Financial Inclusion EDA and Forecasting
This notebook loads the unified dataset, explores Access and Usage indicators, reviews event impacts, and builds a simple forecast for 2025-2027.

In [ ]:
import pandas as pd
import plotly.express as px
from src.data_loader import load_unified_data, load_impact_sheet

data = load_unified_data()
impacts = load_impact_sheet()
observations = data[data['record_type'] == 'observation'].copy()
observations['observation_date'] = pd.to_datetime(observations['observation_date'])

In [1]:
print('Record counts by type')
print(data['record_type'].value_counts())
print()
print('Indicator coverage')
print(observations['indicator_code'].value_counts())
print()
print('Event counts by category')
print(data[data['record_type'] == 'event']['category'].value_counts())

Record counts by type


NameError: name 'data' is not defined

In [ ]:
access = observations[observations['indicator_code'] == 'ACC_OWNERSHIP']
usage = observations[observations['indicator_code'] == 'USG_DIGITAL_PAYMENT']
summary = pd.concat([access[['observation_date','value_numeric']], usage[['observation_date','value_numeric']].rename(columns={'value_numeric':'usage'})], axis=1)
fig = px.line(observations[observations['indicator_code'].isin(['ACC_OWNERSHIP','USG_DIGITAL_PAYMENT'])], x='observation_date', y='value_numeric', color='indicator_code', markers=True, title='Access and Usage Time Series')
fig.show()

In [ ]:
forecast_years = [2025, 2026, 2027]
access_years = access['observation_date'].dt.year.values
usage_years = usage['observation_date'].dt.year.values
access_slope = (access['value_numeric'].iloc[-1] - access['value_numeric'].iloc[-2])
usage_slope = 2.5
access_base = access['value_numeric'].iloc[-1] if len(access) > 0 else 0
usage_base = usage['value_numeric'].iloc[-1] if len(usage) > 0 else 0
access_forecast = [access_base + access_slope * (y - access_years[-1]) for y in forecast_years]
usage_forecast = [usage_base + usage_slope * (y - usage_years[-1]) for y in forecast_years]
forecast_df = pd.DataFrame({'year': forecast_years, 'Access': access_forecast, 'Usage': usage_forecast})
forecast_df